In [0]:
#Setup

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}"

)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"

)

print("Schemas Gold and Silver available")


In [0]:
#Functions

def read_bronze(table_name: str) -> DataFrame:
    """It reads a bronze table"""
    return spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

def save_silver(
    dataframe: DataFrame,
    table_name: str,
) -> None: 
    """It saves a dataframe as a delta table on Silver"""

    target_table = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"

    )

    (   
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(target_table)

    )
    row_count = dataframe.count()

    print(
        f"Table saved: {target_table} "
        f"({row_count:,} rows)"
    )


In [0]:
#CUSTOMERS

#Read the customers_bronze tables
customers_bronze = read_bronze("olist_customers")
display(customers_bronze.limit(5))
customers_bronze.printSchema()

#Clean the data

window_spec = Window.partitionBy("customer_id").orderBy(F.desc("_ingested_at"))

customers_silver = (
    customers_bronze
    .select(
        F.trim("customer_id").alias("customer_id"),
        F.trim("customer_unique_id").alias("customer_unique_id"),
        F.col("customer_zip_code_prefix").cast("int").alias("customer_zip_code_prefix"),
        F.initcap(F.trim("customer_city")).alias("customer_city"),
        F.upper(F.trim("customer_state")).alias("customer_state"),
        F.col("_ingested_at")
    )
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .filter(F.col("customer_id").isNotNull())
)

#Validation

display(customers_silver.limit(10))

print(
    "Bronze rows:",
        customers_bronze.count(),
)
print(
    "Silver rows:",
        customers_silver.count()

)

print(
    "Distinct customer_id's:",
    customers_silver
    .select("customer_id")
    .distinct()
    .count()

)

#Save Customers Silver Table

save_silver(
    customers_silver,
    "customers",

)


In [0]:
#ORDERS

#Read the orders_bronze tables
orders_bronze = read_bronze("olist_orders")
display(orders_bronze.limit(5))
orders_bronze.printSchema()

#Clean the data

window_spec = Window.partitionBy("order_id").orderBy(F.desc("_ingested_at"))

orders_silver = (
    orders_bronze
    .select(
        F.trim("order_id").alias("order_id"),
        F.trim("customer_id").alias("customer_id"),
        F.lower(F.trim("order_status")).alias("order_status"),
        F.to_timestamp(F.trim("order_purchase_timestamp")).alias("purchased_at"),
        F.to_timestamp(F.trim("order_approved_at")).alias("approved_at"),
        F.to_timestamp(F.trim("order_delivered_carrier_date")).alias("delivered_to_carrier_at"),
        F.to_timestamp(F.trim("order_delivered_customer_date")).alias("delivered_to_customer_at"),
        F.to_timestamp(F.trim("order_estimated_delivery_date")).alias("estimated_delivery_at"),
        F.col("_ingested_at"),
    )
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .filter(F.col("order_id").isNotNull())
)

#Validation

display(orders_silver.limit(10))

print(
    "Bronze rows:",
        orders_bronze.count(),
)
print(
    "Silver rows:",
        orders_silver.count()

)

print(
    "Distinct order_id's:",
    orders_silver
    .select("order_id")
    .distinct()
    .count()

)

#Save ORDERS Silver Table

save_silver(
    orders_silver,
    "orders",

)


In [0]:
#ORDER_ITEMS

#Read the order_items bronze tables
order_items_bronze = read_bronze("olist_order_items")
display(order_items_bronze.limit(5))
order_items_bronze.printSchema()

#Clean the data

order_items_silver = (
    order_items_bronze
    .select(
        F.trim("order_id").alias("order_id"),
        F.col("order_item_id").cast("integer").alias("order_item_sequence"),
        F.trim("product_id").alias("product_id"),
        F.trim("seller_id").alias("seller_id"),
        F.to_timestamp("shipping_limit_date").alias("shipping_limit_at"),
        F.round(F.col("price").cast("double"), 2,).alias("item_price"),
        F.round(F.col("freight_value").cast("double"), 2,).alias("freight_value"),
        F.col("_ingested_at"),
    )
    .withColumn("_row_num", F.row_number().over(
        Window.partitionBy("order_id", "order_item_sequence").orderBy(F.desc("_ingested_at"))
    ))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("item_price").isNotNull())
    .filter(F.col("product_id").isNotNull())
)

#Validation

display(order_items_silver.limit(10))

print(
    "Bronze rows:",
        order_items_bronze.count(),
)
print(
    "Silver rows:",
        order_items_silver.count()

)

print(
    "Distinct order_id's:",
    order_items_silver
    .select("order_id")
    .distinct()
    .count()

)

#Save ORDERS Silver Table

save_silver(
    order_items_silver,
    "order_items",

)


In [0]:
#PAYMENTS

payments_bronze = read_bronze("olist_order_payments")
display(payments_bronze.limit(500))

payments_silver = (payments_bronze
    .select(
        F.trim("order_id").alias("order_id"),
        F.col("payment_sequential").cast("integer").alias("payment_sequential"),
        F.lower(F.trim("payment_type")).alias("payment_type"),
        F.round(F.col("payment_value").cast("double"), 2).alias("payment_value"),
    )
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("payment_value") >= 0)
    .dropDuplicates(["order_id", "payment_value", "payment_sequential"])

)

#AGGREGATION: PAYMENTS BY ORDER

payments_by_order = (payments_silver
    .groupBy("order_id")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
        F.max("payment_installments").alias("max_installments"),
        F.sort_array(F.collect_set("payment_type")).alias("payment_methods"),
        F.count("*").alias("payment_record_count"),

    )

)

save_silver(payments_silver, "payments")



In [0]:
%sql
SELECT order_id, payment_type, payment_value
FROM workspace.ecommerce_silver.payments
LIMIT 20;